# 01 · Minimal Kernel：200 行自包含内核

**目标读者**：贡献者、想理解内核的人

**前置条件**：仅 `pandas`、`numpy`、`pyarrow`（写 parquet）与标准库；不 import `finpass`，完全自包含

**核心目的**：本 notebook 要证明三句话 ——

1. 五步流程（检测 → 决策 → 修复 → 复检 → 留痕）在最小规模成立。
2. 每个单元格变更都能追溯到 `rule_id + decision_id`。
3. 按主键对齐的 `changes` 能 replay 出最终 df。

声明：本 notebook 的规则写死，仅教学；生产版规则来自 profile。

**输出目录**：`out/01/`（clean.csv、events.jsonl、changes.parquet、passport.json、diff.html）

**运行时长**：< 10 秒

**验收标准**：

- 全程不 import `finpass`，只依赖 pandas / numpy / pyarrow 与标准库；
- 至少生成 column / strategy / point 三类问题各一个；
- `events.jsonl` 行数 = 回答次数，`changes.parquet` 每行都有 `decision_id`；
- `diff.html` 独立打开，至少 3 种颜色，hover 有 `rule_id`；
- 末尾 replay 断言通过；
- `passport.json` 含 `input_hash`、`output_file_sha256`、`human_decisions_count`；
- 总代码量 ≤ 250 行（不含 markdown）。

## 大纲

1. 导入与固定种子：输出目录、输入 CSV、硬编码主键
2. 三个 dataclass：Question / CellChange / Decision
3. MiniSession 骨架与 `_hash`
4. 注入坏数据：让最小样例能同时触发三类问题（固定行、无随机数）
5. detect：三条规则覆盖 column / strategy / point 三档
6. next / answer：一次一问，事件与单元格变更同时落盘
7. 跑一遍交互：硬编码答案代替 widget，展示三种 action
8. render_diff：最小 HTML 高亮
9. export：五个文件
10. replay 验证：这本 notebook 的测试

In [1]:
import json, uuid, hashlib, datetime
from dataclasses import dataclass, asdict
from pathlib import Path
import pandas as pd, numpy as np

NB_DIR = Path.cwd()
ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
OUT = ROOT / "out" / "01"; OUT.mkdir(parents=True, exist_ok=True)
INPUT = ROOT / "examples" / "pingan_bank_daily.csv"
KEY = "日期"          # 教学版硬编码主键；生产版由 profile 声明
print(f"ROOT={ROOT}")

ROOT=d:\Files\project\finpass


In [2]:
@dataclass
class Question:
    question_id: str
    rule_id: str
    aggregation_level: str          # strategy | column | point
    affected_row_keys: list
    samples: list
    options: list
    default: str


@dataclass
class CellChange:
    row_key: str
    col: str
    before: object
    after: object
    change_type: str                # filled_nan | changed_value | normalized | dropped_row | flagged_kept
    rule_id: str
    decision_id: str


@dataclass
class Decision:
    event_id: str
    ts: str
    question_id: str
    rule_id: str
    action: str                     # answer | skip | batch_apply | undo
    params: dict
    reason: str
    affected_row_keys: list
    data_hash_before: str
    data_hash_after: str

print("dataclass 就绪；CellChange 只用 row_key，不用行号")

dataclass 就绪；CellChange 只用 row_key，不用行号


In [3]:
class MiniSession:
    def __init__(self, df, key):
        self.original = df.copy()
        self.df = df.copy()
        self.key = key
        self.events = []
        self.changes = []
        self.queue = []
        self._answered = set()

    def _hash(self, df):
        return hashlib.sha256(
            pd.util.hash_pandas_object(df, index=True).values.tobytes()
        ).hexdigest()[:16]

print("MiniSession 骨架就绪；_hash 走 pandas hash，不依赖 CSV 序列化细节")

MiniSession 骨架就绪；_hash 走 pandas hash，不依赖 CSV 序列化细节


In [4]:
def build_dirty(base: pd.DataFrame) -> pd.DataFrame:
    """可复现的坏数据注入：只用固定行，无随机数。"""
    dup = base.iloc[[1]].copy()                  # 复制第 2 行 -> 重复主键
    bad = base.iloc[[0]].copy()                  # 复制第 1 行 -> 制造 OHLC 矛盾点
    bad[KEY] = "2024-01-05"
    bad["最高价"], bad["最低价"] = 9.50, 9.90    # 故意 High < Low
    return pd.concat([base, dup, bad], ignore_index=True)

base = pd.read_csv(INPUT)
dirty = build_dirty(base)
print(f"教学输入：{len(dirty)} 行 = 原始 {len(base)} 行 + 注入 2 行（1 个重复主键、1 个矛盾点）")

教学输入：5 行 = 原始 3 行 + 注入 2 行（1 个重复主键、1 个矛盾点）


In [5]:
def detect(self):
    # 规则 1：缺失 token（column 级，整列一个确认问题）
    missing_tokens = ["--", "暂无", "N/A"]
    for col in self.df.columns:
        mask = self.df[col].astype(str).isin(missing_tokens)
        if mask.any():
            self.queue.append(Question(
                question_id=f"q_missing_{col}", rule_id="missing_token",
                aggregation_level="column",
                affected_row_keys=self.df.loc[mask, self.key].tolist(),
                samples=self.df.loc[mask, [self.key, col]].head(5).to_dict("records"),
                options=["填 NaN", "跳过打标"], default="填 NaN"))

    # 规则 2：重复主键（strategy 级，一个策略覆盖全部）
    dup = self.df[self.df.duplicated(self.key, keep=False)]
    if not dup.empty:
        self.queue.append(Question(
            question_id="q_dup_key", rule_id="duplicate_key",
            aggregation_level="strategy",
            affected_row_keys=dup[self.key].unique().tolist(),
            samples=dup[[self.key]].head(5).to_dict("records"),
            options=["保留第一条", "保留最后一条", "跳过打标"], default="保留第一条"))

    # 规则 3：OHLC 矛盾（point 级，逐行问，必须带上下文）
    for _, r in self.df.iterrows():
        if pd.notna(r.get("最高价")) and pd.notna(r.get("最低价")) and r["最高价"] < r["最低价"]:
            self.queue.append(Question(
                question_id=f"q_ohlc_{r[self.key]}", rule_id="ohlc_contradiction",
                aggregation_level="point",
                affected_row_keys=[r[self.key]],
                samples=[{"日期": r[self.key], "最高价": r["最高价"], "最低价": r["最低价"], "收盘价": r.get("收盘价")}],
                options=["标记跳过", "以收盘价为准修正"], default="标记跳过"))

MiniSession.detect = detect

session = MiniSession(dirty, KEY)
session.detect()
print(f"问题数：{len(session.queue)}")
for q in session.queue:
    print(f"  [{q.aggregation_level:8}] {q.rule_id:20} 影响 {len(q.affected_row_keys)} 个主键")

问题数：3
  [column  ] missing_token        影响 1 个主键
  [strategy] duplicate_key        影响 1 个主键
  [point   ] ohlc_contradiction   影响 1 个主键


In [6]:
def next(self):
    for q in self.queue:
        if q.question_id not in self._answered:
            return q
    return None

def answer(self, q, action, params=None, reason=""):
    before_hash = self._hash(self.df)
    decision_id = uuid.uuid4().hex
    changed = []

    if q.rule_id == "missing_token" and action == "answer":
        col = q.question_id.replace("q_missing_", "", 1)
        for rk in q.affected_row_keys:
            idx = self.df.index[self.df[self.key] == rk][0]
            old = self.df.at[idx, col]
            self.df.at[idx, col] = np.nan
            changed.append(CellChange(rk, col, str(old), np.nan, "filled_nan", q.rule_id, decision_id))

    elif q.rule_id == "duplicate_key" and action == "answer":
        keep_first = params.get("keep", "保留第一条") == "保留第一条"
        to_drop = self.df[self.df.duplicated(self.key, keep="first" if keep_first else "last")]
        for _, row in to_drop.iterrows():
            changed.append(CellChange(row[self.key], "__row__",
                                      json.dumps(row.to_dict(), ensure_ascii=False, default=str),
                                      None, "dropped_row", q.rule_id, decision_id))
        self.df = self.df.drop_duplicates(self.key, keep="first" if keep_first else "last")

    elif action == "skip":
        for rk in q.affected_row_keys:
            for col in q.samples[0].keys():
                changed.append(CellChange(rk, col, None, None, "flagged_kept", q.rule_id, decision_id))

    after_hash = self._hash(self.df)
    self.events.append(Decision(decision_id, datetime.datetime.now().isoformat(), q.question_id,
                                q.rule_id, action, params or {}, reason,
                                q.affected_row_keys, before_hash, after_hash))
    self.changes.extend(changed)
    self._answered.add(q.question_id)
    return changed

MiniSession.next = next
MiniSession.answer = answer
print("next / answer 就绪；answer 同时写 events 与 changes，二者靠 decision_id 关联")

next / answer 就绪；answer 同时写 events 与 changes，二者靠 decision_id 关联


In [7]:
q1 = session.next(); print("Q1:", q1.question_id); session.answer(q1, "answer", {}, "缺失 token 统一填 NaN")
q2 = session.next(); print("Q2:", q2.question_id); session.answer(q2, "answer", {"keep": "保留第一条"}, "重复主键保留第一条")
q3 = session.next(); print("Q3:", q3.question_id); session.answer(q3, "skip", reason="疑似数据源错误，保留打标")

print("剩余问题：", session.next())
print(f"events={len(session.events)} changes={len(session.changes)} "
      f"unresolved={sum(1 for c in session.changes if c.change_type == 'flagged_kept')}")

Q1: q_missing_开盘价
Q2: q_dup_key
Q3: q_ohlc_2024-01-05
剩余问题： None
events=3 changes=6 unresolved=4


In [8]:
CHANGE_COLORS = {"filled_nan": "#d4f8d4", "changed_value": "#fff3cd",
                 "normalized": "#d1ecf1", "dropped_row": "#f8d7da", "flagged_kept": "#ffe8b3"}

def render_diff(orig, final, changes, path):
    lookup = {(c.row_key, c.col): c for c in changes if c.change_type != "dropped_row"}
    rows = []
    for _, r in final.iterrows():
        cells = []
        for col in final.columns:
            c = lookup.get((r[KEY], col))
            style = f"background:{CHANGE_COLORS[c.change_type]}" if c else ""
            title = f"{c.rule_id} / {c.decision_id}" if c else ""
            val = "" if pd.isna(r[col]) else r[col]
            cells.append(f'<td style="{style}" title="{title}">{val}</td>')
        rows.append(f"<tr>{''.join(cells)}</tr>")
    head = "".join(f"<th>{c}</th>" for c in final.columns)
    drop_style = CHANGE_COLORS["dropped_row"]
    for c in [c for c in changes if c.change_type == "dropped_row"]:
        rows.append(f'<tr style="background:{drop_style}" title="{c.rule_id} / {c.decision_id}">'
                    f'<td>{c.row_key}（已删除）</td>'
                    f'<td colspan="{len(final.columns) - 1}">{c.before}</td></tr>')
    html = f"<html><body><table border=1><tr>{head}</tr>{''.join(rows)}</table></body></html>"
    Path(path).write_text(html, encoding="utf-8")

In [9]:
def export(self, outdir):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    self.df.to_csv(outdir / "clean.csv", index=False)

    with open(outdir / "events.jsonl", "w", encoding="utf-8") as f:
        for e in self.events:
            f.write(json.dumps(asdict(e), ensure_ascii=False) + "\n")

    pd.DataFrame([asdict(c) for c in self.changes]).to_parquet(outdir / "changes.parquet", index=False)

    passport = {
        "input_hash": self._hash(self.original),
        "output_file_sha256": hashlib.sha256((outdir / "clean.csv").read_bytes()).hexdigest(),
        "profile": "minimal_kernel_v0",
        "human_decisions_count": len([e for e in self.events if e.action != "auto"]),
        "unresolved_issues_count": sum(1 for c in self.changes if c.change_type == "flagged_kept"),
        "auto_only": all(e.action == "auto" for e in self.events),
        "reproduce": "notebooks/01_minimal_kernel.ipynb",
    }
    (outdir / "passport.json").write_text(json.dumps(passport, ensure_ascii=False, indent=2), encoding="utf-8")
    render_diff(self.original, self.df, self.changes, outdir / "diff.html")

MiniSession.export = export
session.export(OUT)
print(sorted(p.name for p in OUT.iterdir()))

['changes.parquet', 'clean.csv', 'diff.html', 'events.jsonl', 'passport.json']


In [10]:
def replay(input_df, changes_path, key):
    df = input_df.copy()
    for ch in pd.read_parquet(changes_path).itertuples():
        if ch.change_type == "flagged_kept":
            continue                      # 打标不改值：replay 必须跳过
        if ch.change_type == "dropped_row":
            idx = df.index[df[key] == ch.row_key]
            if len(idx):
                df = df.drop(idx[-1])     # keep="first" 的镜像：删该主键的末次出现
            continue
        idx = df.index[df[key] == ch.row_key]
        if len(idx) == 0:
            continue
        df.at[idx[0], ch.col] = np.nan if pd.isna(ch.after) else ch.after
    return df.reset_index(drop=True)

rebuilt = replay(dirty, OUT / "changes.parquet", KEY)
expected = session.df.reset_index(drop=True)
pd.testing.assert_frame_equal(rebuilt, expected, check_dtype=False)
print("✅ replay 一致")

✅ replay 一致


## 交给生产版的 5 条约束

从本 notebook 到 `src/finpass/` 必须升级的点：

1. 规则不能写死 → `engine.dispatch(rule_id)` 从 profile 读。
2. `_hash` 要包含 profile 版本和规则版本。
3. `dropped_row` 必须记录被删行的完整 before 值与稳定行标识 —— 本 notebook 靠主键末次出现推断，同一主键重复三次就会错。
4. `render_diff` 要做主键对齐的完整表格重建（删行后行号不能错位），并支持大表折叠。
5. `answer` 要支持 `batch_apply` 展开，且一个 decision 关联多条 change。

另一处必须收口的语义：`flagged_kept` 的 `after` 是空值，replay 必须显式跳过，否则会把“打标保留”误当成“清空该格”。